In [1]:
# 1. Tải mã nguồn PaddleOCR và di chuyển vào thư mục
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 2. Cài đặt thư viện lõi (Bản có hỗ trợ GPU)
!pip install paddlepaddle-gpu -U
!pip install -r requirements.txt

%cd ..

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 337229, done.
remote: Counting objects: 100% (954/954), done.
remote: Compressing objects: 100% (319/319), done.
remote: Total 337229 (delta 756), reused 635 (delta 635), pack-reused 336275 (from 3)
Receiving objects: 100% (337229/337229), 1.79 GiB | 32.45 MiB/s, done.
Resolving deltas: 100% (267219/267219), done.
/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 2.4 MB/s eta 0:00:00:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 6.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.4 MB/s eta 0:00:00:00:01
/kaggle/working


In [2]:
# Tạo file từ điển (Bảng chữ cái biển số)
alphabet = "0123456789ABCDEFGHKLMNPQRSTUVXYZĐ"

dict_path = "/kaggle/working/PaddleOCR/ppocr/utils/plate_dict.txt"
with open(dict_path, "w", encoding="utf-8") as f:
    for char in alphabet:
        f.write(char + "\n")
print("Done")

Done


In [ ]:
%%writefile /kaggle/working/PaddleOCR/configs/rec/multi_language/rec_multi_language_lite_train.yml
Global:
  use_gpu: True
  epoch_num: 500
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec_multi_language_lite
  save_epoch_step: 3
  # evaluation is run every 5000 iterations after the 4000th iteration
  eval_batch_step: [0, 2000]
  # if pretrained_model is saved in static mode, load_static_weights must set to True
  cal_metric_during_train: True
  pretrained_model:
  checkpoints:
  save_inference_dir:
  use_visualdl: False
  infer_img:
  # for data or label process
  character_dict_path: /kaggle/working/PaddleOCR/ppocr/utils/plate_dict.txt
  # Set the language of training, if set, select the default dictionary file
  character_type:
  max_text_length: 11
  infer_mode: False
  use_space_char: False


Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
  regularizer:
    name: 'L2'
    factor: 0.00001

Architecture:
  model_type: rec
  algorithm: CRNN
  Transform:
  Backbone:
    name: MobileNetV3
    scale: 0.5
    model_name: large
  Neck:
    name: SequenceEncoder
    encoder_type: rnn
    hidden_size: 96
  Head:
    name: CTCHead
    fc_decay: 0.00001

Loss:
  name: CTCLoss

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /kaggle/input/datasets/tada924/plate-vietnam/dataset_crnn_v5/train/images
    label_file_list: ["/kaggle/input/datasets/tada924/plate-vietnam/dataset_crnn_v5/train/labels.txt"]
    transforms:
      - DecodeImage: # load image
          img_mode: BGR
          channel_first: False
      - RecAug:
      - CTCLabelEncode: # Class handling label
      - RecResizeImg:
          image_shape: [3, 32, 320]
      - KeepKeys:
          keep_keys: ['image', 'label', 'length'] # dataloader will return list in this order
  loader:
    shuffle: True
    batch_size_per_card: 192
    drop_last: True
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /kaggle/input/datasets/tada924/plate-vietnam/dataset_crnn_v5/val/images
    label_file_list: ["/kaggle/input/datasets/tada924/plate-vietnam/dataset_crnn_v5/val/labels.txt"]
    transforms:
      - DecodeImage: # load image
          img_mode: BGR
          channel_first: False
      - CTCLabelEncode: # Class handling label
      - RecResizeImg:
          image_shape: [3, 32, 320]
      - KeepKeys:
          keep_keys: ['image', 'label', 'length'] # dataloader will return list in this order
  loader:
    shuffle: False
    drop_last: False
    batch_size_per_card: 128
    num_workers: 4

Overwriting /kaggle/working/PaddleOCR/configs/rec/multi_language/rec_multi_language_lite_train.yml


In [4]:
%cd /kaggle/working/PaddleOCR
!python tools/train.py -c configs/rec/multi_language/rec_multi_language_lite_train.yml

/kaggle/working/PaddleOCR
Skipping import of the encryption module.
[2026/04/24 02:56:49] ppocr INFO: Architecture : 
[2026/04/24 02:56:49] ppocr INFO:     Backbone : 
[2026/04/24 02:56:49] ppocr INFO:         model_name : large
[2026/04/24 02:56:49] ppocr INFO:         name : MobileNetV3
[2026/04/24 02:56:49] ppocr INFO:         scale : 0.5
[2026/04/24 02:56:49] ppocr INFO:     Head : 
[2026/04/24 02:56:49] ppocr INFO:         fc_decay : 1e-05
[2026/04/24 02:56:49] ppocr INFO:         name : CTCHead
[2026/04/24 02:56:49] ppocr INFO:     Neck : 
[2026/04/24 02:56:49] ppocr INFO:         encoder_type : rnn
[2026/04/24 02:56:49] ppocr INFO:         hidden_size : 96
[2026/04/24 02:56:49] ppocr INFO:         name : SequenceEncoder
[2026/04/24 02:56:49] ppocr INFO:     Transform : None
[2026/04/24 02:56:49] ppocr INFO:     algorithm : CRNN
[2026/04/24 02:56:49] ppocr INFO:     model_type : rec
[2026/04/24 02:56:49] ppocr INFO: Eval : 
[2026/04/24 02:56:49] ppocr INFO:     dataset : 
[2026/0

In [5]:
!pip uninstall paddlepaddle paddlepaddle-gpu -y
!python -m pip install paddlepaddle-gpu==3.0.0rc1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

Found existing installation: paddlepaddle-gpu 2.6.2
Uninstalling paddlepaddle-gpu-2.6.2:
  Successfully uninstalled paddlepaddle-gpu-2.6.2
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 1.1 MB/s eta 0:00:00:00:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 10.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 10.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 1.5 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.0 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 4.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 8.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 6.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 4

In [6]:
%cd /kaggle/working/PaddleOCR
!python tools/train.py -c configs/rec/multi_language/rec_multi_language_lite_train.yml

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/04/24 03:04:43] ppocr INFO: Architecture : 
[2026/04/24 03:04:43] ppocr INFO:     Backbone : 
[2026/04/24 03:04:43] ppocr INFO:         model_name : large
[2026/04/24 03:04:43] ppocr INFO:         name : MobileNetV3
[2026/04/24 03:04:43] ppocr INFO:         scale : 0.5
[2026/04/24 03:04:43] ppocr INFO:     Head : 
[2026/04/24 03:04:43] ppocr INFO:         fc_decay : 1e-05
[2026/04/24 03:04:43] ppocr INFO:         name : CTCHead
[2026/04/24 03:04:43] ppocr INFO:     Neck : 
[2026/04/24 03:04:43] ppocr INFO:         encoder_type : rnn
[2026/04/24 03:04:43] ppocr INFO:         hidden_size : 96
[2026

In [7]:
!pip install paddlepaddle==3.0.0b2
!pip install paddle2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 19.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 36.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.8/372.8 kB 19.8 MB/s eta 0:00:00
  Attempting uninstall: onnx
    Found existing installation: onnx 1.20.1
    Uninstalling onnx-1.20.1:
      Successfully uninstalled onnx-1.20.1


In [ ]:
%cd /kaggle/working/PaddleOCR
!python tools/export_model.py \
    -c configs/rec/multi_language/rec_multi_language_lite_train.yml \
    -o Global.pretrained_model=./output/rec_multi_language_lite/best_accuracy \
    Global.save_inference_dir=./output/export_model \
    Global.character_dict_path=./ppocr/utils/plate_dict.txt \
    Architecture.Head.return_all=false

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:686: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/04/24 07:50:07] ppocr INFO: load pretrain successful from ./output/rec_multi_language_lite/best_accuracy
[2026/04/24 07:50:07] ppocr INFO: Export inference config file to ./output/export_model/inference.yml
Skipping import of the encryption module
[2026/04/24 07:50:09] ppocr INFO: inference model is saved to ./output/export_model/inference


In [9]:
!paddle2onnx \
    --model_dir ./output/export_model \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file ./output/export_model/rec_model.onnx \
    --opset_version 12 \
    --enable_onnx_checker True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:686: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[Paddle2ONNX] Start parsing the Paddle model file...
[Paddle2ONNX] Use opset_version = 12 for ONNX export.
[Paddle2ONNX] PaddlePaddle model is exported as ONNX format now.
2026-04-24 07:50:25 [INFO]	Try to perform constant folding on the ONNX model with Polygraphy.
[W] 'colored' module is not installed, will not use colors when logging. To enable colors, please install the 'colored' module: python3 -m pip install colored
[I] Module: 'onnxruntime' is required, but not installed. Attempting to install now.
[I] Running installation command: /usr/bin/python3 -m pip install onnxruntime>=1.10.0
[I] Folding Constants | Pass 1
[I] Module: 'onnx_graphsurgeon' is required, but n

In [10]:
%cd /kaggle/working
!zip -r full_kaggle_workspace.zip ./*

/kaggle/working
  adding: PaddleOCR/ (stored 0%)
  adding: PaddleOCR/configs/ (stored 0%)
  adding: PaddleOCR/configs/sr/ (stored 0%)
  adding: PaddleOCR/configs/sr/sr_telescope.yml (deflated 58%)
  adding: PaddleOCR/configs/sr/sr_tsrn_transformer_strock.yml (deflated 60%)
  adding: PaddleOCR/configs/rec/ (stored 0%)
  adding: PaddleOCR/configs/rec/rec_mv3_tps_bilstm_ctc.yml (deflated 59%)
  adding: PaddleOCR/configs/rec/rec_mv3_none_none_ctc.yml (deflated 59%)
  adding: PaddleOCR/configs/rec/rec_resnet_rfl_att.yml (deflated 61%)
  adding: PaddleOCR/configs/rec/rec_vit_parseq.yml (deflated 59%)
  adding: PaddleOCR/configs/rec/rec_r31_sar.yml (deflated 60%)
  adding: PaddleOCR/configs/rec/rec_svtrnet.yml (deflated 60%)
  adding: PaddleOCR/configs/rec/rec_r45_abinet.yml (deflated 60%)
  adding: PaddleOCR/configs/rec/SVTRv2/ (stored 0%)
  adding: PaddleOCR/configs/rec/SVTRv2/ch_RepSVTR_rec.yml (deflated 61%)
  adding: PaddleOCR/configs/rec/SVTRv2/ch_SVTRv2_rec_distillation.yml (deflated 6